# Quick Price Range Check

Goal: load market data, find the observed minimum and maximum price, and inspect nearby values for a quick judgment on band tightness.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load data and choose a price column

In [6]:
candidate_paths = [
    # Path("data/market_data.csv"),
    Path("data/market_data_large.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find data/market_data.csv or data/market_data_large.csv")

df = pd.read_csv(csv_path)
print(f"Using file: {csv_path}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")

common_price_names = ["price", "px", "mid", "close", "last"]
name_map = {c.lower(): c for c in df.columns}

price_col = next((name_map[n] for n in common_price_names if n in name_map), None)
if price_col is None:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_cols:
        raise ValueError("No numeric columns found to infer a price column")
    price_col = numeric_cols[0]

print(f"Selected price column: {price_col}")

Using file: data\market_data_large.csv
Rows: 39,662,212 | Columns: 8
Selected price column: price


In [7]:
prices_raw = pd.to_numeric(df[price_col], errors="coerce")
prices = prices_raw[(prices_raw > 0) & np.isfinite(prices_raw)].dropna()

if prices.empty:
    raise ValueError("No valid positive prices found")

pmin = float(prices.min())
pmax = float(prices.max())
spread = pmax - pmin

print(f"Observed min price: {pmin}")
print(f"Observed max price: {pmax}")
print(f"Absolute spread: {spread}")
print(f"Relative spread vs min: {(spread / pmin) * 100:.4f}%")

unique_prices = np.sort(prices.unique())
print(f"Unique prices: {len(unique_prices):,}")

print("\n10 smallest unique prices:")
print(unique_prices[:10])

print("\n10 largest unique prices:")
print(unique_prices[-10:])

Observed min price: 5929.0
Observed max price: 12450.0
Absolute spread: 6521.0
Relative spread vs min: 109.9848%
Unique prices: 6,512

10 smallest unique prices:
[5929. 5933. 5934. 5936. 5938. 5939. 5940. 5941. 5942. 5943.]

10 largest unique prices:
[12436. 12437. 12438. 12439. 12440. 12443. 12444. 12446. 12448. 12450.]
